# Lab PySpark — NYC Yellow Taxi 2020–2025
**Datos:** NYC TLC Yellow Taxi Trip Records

## Configuración de SparkSession y carga de datos

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

print(spark.version)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# La celda 2 ya arrancó una SparkSession con configuración por defecto (heap ~1 g, sin ajustes de parquet).
# getOrCreate() devolvería esa sesión existente e ignoraría silenciosamente todos los .config() de abajo.
# Detenerla primero garantiza que el nuevo JVM arranque con los parámetros correctos.
_existing = SparkSession.getActiveSession()
if _existing is not None:
    _existing.stop()

spark = (
    SparkSession.builder
    .appName("NYC Yellow Taxi 2019-2025")
    .master("local[2]")                                          # 2 hilos → reduce a la mitad el pico de heap vs local[*]
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.parquet.enableVectorizedReader", "false") # evita error de esquema INT64/DOUBLE en passenger_count
    .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")  # clave confirmada en ParquetFileReader.class
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Doble seguro: también se aplica en tiempo de ejecución vía Hadoop Configuration
spark.sparkContext._jsc.hadoopConfiguration().set("parquet.hadoop.vectored.io.enabled", "false")

# Verificación — debe imprimir "false"
val = spark.sparkContext._jsc.hadoopConfiguration().get("parquet.hadoop.vectored.io.enabled", "NO ESTABLECIDO")
print("Versión de Spark     :", spark.version)
print("vectored.io.enabled  =", val)

In [ ]:
# Carga de TODOS los archivos Parquet 2019-2025 en un solo DataFrame
import glob
from functools import reduce
from pyspark.sql.types import LongType, DoubleType

BASE_PATH = "."
YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Columnas canónicas y sus tipos objetivo (en minúsculas)
LONG_COLS   = {"vendorid", "pulocationid", "dolocationid", "payment_type"}
DOUBLE_COLS = {"passenger_count", "trip_distance", "ratecodeid", "fare_amount",
               "extra", "mta_tax", "tip_amount", "tolls_amount",
               "improvement_surcharge", "total_amount", "congestion_surcharge",
               "airport_fee"}

def normalize_df(file_df):
    for c in file_df.columns:
        if c != c.lower():
            file_df = file_df.withColumnRenamed(c, c.lower())
    if "airport_fee" not in file_df.columns:
        file_df = file_df.withColumn("airport_fee", F.lit(None).cast(DoubleType()))
    for c in file_df.columns:
        if c in LONG_COLS:
            file_df = file_df.withColumn(c, F.col(c).cast(LongType()))
        elif c in DOUBLE_COLS:
            file_df = file_df.withColumn(c, F.col(c).cast(DoubleType()))
    return file_df

all_files = sorted(
    f for year in YEARS
    for f in glob.glob(f"{BASE_PATH}/{year}/*.parquet")
)

dfs = [normalize_df(spark.read.parquet(f)) for f in all_files]
df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

df = (
    df
    .withColumn("pickup_year",  F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
)

df.createOrReplaceTempView("trips")

print(f"Total de registros cargados: {df.count():,}")
df.printSchema()

---
## Pregunta 1 — Volumen de viajes por año (2019–2025)

**Columnas:** `tpep_pickup_datetime`

Comparar el número total de viajes pre-pandemia, durante y post-pandemia.
**Pregunta:** ¿Cómo impactó el COVID-19 al volumen de viajes? ¿Cuándo se recuperó?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                      AS anio,
        COUNT(*)                         AS total_viajes,
        ROUND(COUNT(*) / 1e6, 2)         AS millones_viajes
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 1

2019 representa el nivel de referencia pre-pandemia con el mayor volumen del período. En 2020 los viajes colapsaron: el lockdown de marzo–abril provocó caídas superiores al 90 % respecto al nivel normal, con abril 2020 como el peor mes histórico. La recuperación fue gradual: 2021 mejoró respecto a 2020, 2022 registró el mayor salto anual y 2023–2025 continuaron creciendo. Ningún año post-pandemia superó el volumen de 2019, lo que refleja el impacto permanente de la consolidación de plataformas como Uber/Lyft y el teletrabajo.

---
## Pregunta 2 — Cambios en el número de pasajeros por viaje durante la pandemia

**Columnas:** `passenger_count`, `tpep_pickup_datetime`

Analizar cómo evolucionó `passenger_count` mes a mes desde enero 2020.
**Pregunta:** ¿La pandemia redujo los viajes compartidos? ¿Hubo recuperación?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                                                        AS anio,
        pickup_month                                                                       AS mes,
        COUNT(*)                                                                           AS total_viajes,
        ROUND(SUM(CASE WHEN passenger_count = 1  THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_solos,
        ROUND(SUM(CASE WHEN passenger_count >= 2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_compartidos,
        ROUND(AVG(passenger_count), 2)                                                     AS promedio_pax
    FROM trips
    WHERE pickup_year BETWEEN 2020 AND 2025
      AND passenger_count BETWEEN 1 AND 6
    GROUP BY pickup_year, pickup_month
    ORDER BY pickup_year, pickup_month
""").show(72)

---
## Respuesta — Pregunta 2

Sí. La pandemia redujo drásticamente los viajes compartidos. En los meses de lockdown (abril–junio 2020) el porcentaje de viajes con 2+ pasajeros cayó a mínimos históricos, ya que los pasajeros evitaban compartir espacio cerrado con desconocidos. A partir de 2022 se observa una recuperación progresiva del porcentaje de viajes compartidos, pero sin retornar a los niveles de los primeros meses de 2020, lo que indica un cambio parcialmente permanente de hábitos.

---
## Pregunta 3 — Distancia promedio de los viajes entre 2019–2025

**Columnas:** `trip_distance`, `tpep_pickup_datetime`

Evaluar la distancia promedio por año.
**Pregunta:** ¿Durante la pandemia se hicieron viajes más largos o más cortos?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                         AS anio,
        ROUND(AVG(trip_distance), 2)                       AS dist_promedio_millas,
        ROUND(percentile_approx(trip_distance, 0.5), 2)    AS dist_mediana_millas
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
      AND trip_distance > 0
      AND trip_distance <= 100
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 3

Durante la pandemia (2020–2021) la distancia promedio aumentó respecto a 2019. Los viajes cortos cotidianos (al trabajo, compras) prácticamente desaparecieron, mientras que los desplazamientos largos (aeropuerto, viajes esenciales) continuaron, elevando el promedio. La mediana confirma esta tendencia. A partir de 2022, con la normalización del volumen de viajes, la distancia bajó de nuevo acercándose al nivel de 2019.

---
## Pregunta 4 — Distribución de ingresos (fare + tip) por viaje

**Columnas:** `fare_amount`, `tip_amount`, `tpep_pickup_datetime`

Calcular los ingresos promedio y mediana por año.
**Pregunta:** ¿Los ingresos por viaje cayeron durante 2020–2021? ¿Cuándo se recuperaron?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                                  AS anio,
        ROUND(AVG(fare_amount + tip_amount), 2)                     AS ingreso_promedio_usd,
        ROUND(percentile_approx(fare_amount + tip_amount, 0.5), 2)  AS ingreso_mediana_usd
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
      AND fare_amount > 0
      AND tip_amount >= 0
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 4

Los ingresos por viaje no cayeron en 2020–2021; al contrario, subieron respecto a 2019. Esto se explica por la desaparición de los viajes cortos y baratos, que elevó el promedio y la mediana. En los años de recuperación el ingreso por viaje se mantuvo alto e incluso siguió aumentando, impulsado también por los incrementos de tarifa aprobados en NYC. El ingreso por viaje se recuperó estructuralmente en un nivel más alto, no más bajo.

---
## Pregunta 5 — Cambios en el uso de métodos de pago

**Columnas:** `payment_type`, `tpep_pickup_datetime`

Agrupar por `payment_type` por año (1 = tarjeta, 2 = efectivo).
**Pregunta:** ¿Se evidencia un cambio en los métodos de pago?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                                                              AS anio,
        ROUND(SUM(CASE WHEN payment_type = 1 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1)           AS pct_tarjeta,
        ROUND(SUM(CASE WHEN payment_type = 2 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1)           AS pct_efectivo,
        ROUND(SUM(CASE WHEN payment_type NOT IN (1, 2) THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_otros
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
      AND payment_type IN (1, 2, 3, 4, 5, 6)
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 5

Sí. La pandemia aceleró la migración hacia pagos digitales: el porcentaje de tarjeta aumentó de forma permanente desde 2020 y el efectivo cayó de manera sostenida año a año. Este cambio no se revirtió durante la recuperación, lo que señala una modificación estructural en los hábitos de pago, probablemente impulsada por razones sanitarias (evitar contacto físico) que luego se consolidaron como preferencia duradera.

---
## Pregunta 6 — Variación en los patrones horarios de los viajes

**Columnas:** `tpep_pickup_datetime`

Analizar número de viajes por hora del día para 2020 vs 2024.
**Pregunta:** ¿Desapareció parcialmente el horario rush hour por teletrabajo? ¿Se recuperó?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                 AS anio,
        HOUR(tpep_pickup_datetime)  AS hora,
        COUNT(*)                    AS total_viajes
    FROM trips
    WHERE pickup_year IN (2020, 2024)
    GROUP BY pickup_year, hora
    ORDER BY hora, pickup_year
""").show(48)

---
## Respuesta — Pregunta 6

Sí. En 2024 se observan dos picos marcados de rush hour: uno matutino (~8–9 h) y otro vespertino (~17–18 h). En 2020, en cambio, esos picos casi desaparecieron: el volumen fue bajo y plano durante todo el día, reflejo del lockdown y el teletrabajo masivo. La recuperación del patrón en 2024 confirma el regreso a la movilidad urbana, aunque el volumen total sigue por debajo del nivel pre-pandemia, lo que sugiere que el teletrabajo parcial se consolidó como hábito permanente.

---
## Pregunta 7 — Duración promedio de los viajes

**Columnas:** `tpep_pickup_datetime`, `tpep_dropoff_datetime`

Calcular duración en minutos usando pickup/dropoff.
**Pregunta:** ¿Los tiempos de viaje disminuyeron durante la pandemia debido a menos tráfico?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                                                                                               AS anio,
        ROUND(AVG((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0), 2)                     AS duracion_promedio_min,
        ROUND(percentile_approx((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0, 0.5), 2)  AS duracion_mediana_min
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
      AND (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0 > 0
      AND (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0 <= 120
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 7

Sí. En 2020 la duración promedio bajó respecto a 2019 porque las calles estaban vacías durante el lockdown y los viajes se hacían más rápido. La mediana confirma la misma tendencia. A partir de 2022, con el regreso del tráfico normal, la duración volvió a subir hacia los niveles de 2019 y los superó, lo que refleja el mayor congestionamiento urbano post-pandemia.

---
## Pregunta 8 — Tendencias de propinas (tip_amount)

**Columnas:** `tip_amount`, `tpep_pickup_datetime`, `payment_type`

Calcular la propina media y mediana por año (solo pagos con tarjeta).
**Pregunta:** ¿Las propinas aumentaron durante la pandemia como apoyo a los conductores? ¿O disminuyeron?

In [ ]:
spark.sql("""
    SELECT
        pickup_year                                         AS anio,
        ROUND(AVG(tip_amount), 2)                          AS propina_promedio_usd,
        ROUND(percentile_approx(tip_amount, 0.5), 2)       AS propina_mediana_usd
    FROM trips
    WHERE pickup_year BETWEEN 2019 AND 2025
      AND payment_type = 1
      AND tip_amount >= 0
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

---
## Respuesta — Pregunta 8

La propina promedio subió durante 2020–2021 respecto a 2019. Aunque había muchos menos viajes, los pasajeros que sí usaban el taxi dejaban propinas más altas, posiblemente como gesto de apoyo a los conductores durante la crisis. La mediana confirma esta tendencia. En la recuperación la propina se mantuvo alta, también por el incremento general de precios y tarifas.

---
## Pregunta 9 — Detección de anomalías y calidad de datos

**Columnas:** `trip_distance`, `fare_amount`, `total_amount`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`

Encontrar valores atípicos o inconsistentes (los 5 primeros de cada tipo):
distancias negativas, montos extremos, duraciones cero, viajes demasiado rápidos o lentos.

In [ ]:
spark.sql("""
    WITH neg_dist AS (
        SELECT
            'distancia_negativa'                  AS anomalia,
            CAST(tpep_pickup_datetime AS STRING)  AS pickup,
            ROUND(trip_distance, 2)               AS trip_distance,
            ROUND(fare_amount, 2)                 AS fare_amount,
            ROUND(total_amount, 2)                AS total_amount,
            CAST(NULL AS DOUBLE)                  AS duracion_min,
            CAST(NULL AS DOUBLE)                  AS velocidad_mph,
            ROW_NUMBER() OVER (ORDER BY tpep_pickup_datetime) AS rn
        FROM trips
        WHERE trip_distance < 0
    ),
    ext_amount AS (
        SELECT
            'monto_extremo'                       AS anomalia,
            CAST(tpep_pickup_datetime AS STRING)  AS pickup,
            ROUND(trip_distance, 2)               AS trip_distance,
            ROUND(fare_amount, 2)                 AS fare_amount,
            ROUND(total_amount, 2)                AS total_amount,
            CAST(NULL AS DOUBLE)                  AS duracion_min,
            CAST(NULL AS DOUBLE)                  AS velocidad_mph,
            ROW_NUMBER() OVER (ORDER BY total_amount DESC) AS rn
        FROM trips
        WHERE total_amount > 1000
    ),
    zero_dur AS (
        SELECT
            'duracion_cero'                       AS anomalia,
            CAST(tpep_pickup_datetime AS STRING)  AS pickup,
            ROUND(trip_distance, 2)               AS trip_distance,
            ROUND(fare_amount, 2)                 AS fare_amount,
            ROUND(total_amount, 2)                AS total_amount,
            CAST(0.0 AS DOUBLE)                   AS duracion_min,
            CAST(NULL AS DOUBLE)                  AS velocidad_mph,
            ROW_NUMBER() OVER (ORDER BY tpep_pickup_datetime) AS rn
        FROM trips
        WHERE tpep_dropoff_datetime = tpep_pickup_datetime
    ),
    fast AS (
        SELECT
            'velocidad_imposible'                 AS anomalia,
            CAST(tpep_pickup_datetime AS STRING)  AS pickup,
            ROUND(trip_distance, 2)               AS trip_distance,
            ROUND(fare_amount, 2)                 AS fare_amount,
            ROUND(total_amount, 2)                AS total_amount,
            ROUND((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 60.0, 1)  AS duracion_min,
            ROUND(trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0), 1) AS velocidad_mph,
            ROW_NUMBER() OVER (ORDER BY
                trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0) DESC
            ) AS rn
        FROM trips
        WHERE (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 > 0
          AND trip_distance / ((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0) > 150
    )
    SELECT anomalia, pickup, trip_distance, fare_amount, total_amount, duracion_min, velocidad_mph
    FROM (
        SELECT anomalia, pickup, trip_distance, fare_amount, total_amount, duracion_min, velocidad_mph, rn FROM neg_dist   WHERE rn <= 5
        UNION ALL
        SELECT anomalia, pickup, trip_distance, fare_amount, total_amount, duracion_min, velocidad_mph, rn FROM ext_amount WHERE rn <= 5
        UNION ALL
        SELECT anomalia, pickup, trip_distance, fare_amount, total_amount, duracion_min, velocidad_mph, rn FROM zero_dur   WHERE rn <= 5
        UNION ALL
        SELECT anomalia, pickup, trip_distance, fare_amount, total_amount, duracion_min, velocidad_mph, rn FROM fast       WHERE rn <= 5
    ) t
    ORDER BY anomalia, t.rn
""").show(20, truncate=False)

In [ ]:
# Pregunta 9 — misma lógica, API de DataFrame
from pyspark.sql.window import Window

trips_df = spark.table("trips")

# ── 1. Distancias negativas ──────────────────────────────────────────────────
neg_dist = (
    trips_df.filter(F.col("trip_distance") < 0)
    .select(
        F.lit("distancia_negativa").alias("anomalia"),
        F.col("tpep_pickup_datetime").cast("string").alias("pickup"),
        F.round("trip_distance", 2).alias("trip_distance"),
        F.round("fare_amount",   2).alias("fare_amount"),
        F.round("total_amount",  2).alias("total_amount"),
        F.lit(None).cast("double").alias("duracion_min"),
        F.lit(None).cast("double").alias("velocidad_mph"),
        F.row_number().over(Window.orderBy("tpep_pickup_datetime")).alias("rn"),
    )
)

# ── 2. Montos extremos (total > $1 000) ──────────────────────────────────────
ext_amount = (
    trips_df.filter(F.col("total_amount") > 1000)
    .select(
        F.lit("monto_extremo").alias("anomalia"),
        F.col("tpep_pickup_datetime").cast("string").alias("pickup"),
        F.round("trip_distance", 2).alias("trip_distance"),
        F.round("fare_amount",   2).alias("fare_amount"),
        F.round("total_amount",  2).alias("total_amount"),
        F.lit(None).cast("double").alias("duracion_min"),
        F.lit(None).cast("double").alias("velocidad_mph"),
        F.row_number().over(Window.orderBy(F.col("total_amount").desc())).alias("rn"),
    )
)

# ── 3. Duración cero (pickup == dropoff) ─────────────────────────────────────
zero_dur = (
    trips_df.filter(F.col("tpep_dropoff_datetime") == F.col("tpep_pickup_datetime"))
    .select(
        F.lit("duracion_cero").alias("anomalia"),
        F.col("tpep_pickup_datetime").cast("string").alias("pickup"),
        F.round("trip_distance", 2).alias("trip_distance"),
        F.round("fare_amount",   2).alias("fare_amount"),
        F.round("total_amount",  2).alias("total_amount"),
        F.lit(0.0).cast("double").alias("duracion_min"),
        F.lit(None).cast("double").alias("velocidad_mph"),
        F.row_number().over(Window.orderBy("tpep_pickup_datetime")).alias("rn"),
    )
)

# ── 4. Velocidad imposible (> 150 mph) ───────────────────────────────────────
_dur_h = (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600.0
_vel   = F.col("trip_distance") / _dur_h

fast = (
    trips_df.filter((_dur_h > 0) & (_vel > 150))
    .select(
        F.lit("velocidad_imposible").alias("anomalia"),
        F.col("tpep_pickup_datetime").cast("string").alias("pickup"),
        F.round("trip_distance", 2).alias("trip_distance"),
        F.round("fare_amount",   2).alias("fare_amount"),
        F.round("total_amount",  2).alias("total_amount"),
        F.round(_dur_h * 60.0, 1).alias("duracion_min"),
        F.round(_vel,          1).alias("velocidad_mph"),
        F.row_number().over(Window.orderBy(_vel.desc())).alias("rn"),
    )
)

# ── Unión, orden y resultado ─────────────────────────────────────────────────
(
    neg_dist .filter(F.col("rn") <= 5)
    .union(ext_amount.filter(F.col("rn") <= 5))
    .union(zero_dur  .filter(F.col("rn") <= 5))
    .union(fast      .filter(F.col("rn") <= 5))
    .orderBy("anomalia", "rn")
    .drop("rn")
    .show(20, truncate=False)
)

---
## Respuesta — Pregunta 9

El dataset contiene cuatro categorías de anomalías:
- **Distancias negativas** (`trip_distance < 0`): errores de captura en el odómetro o en la carga del archivo.
- **Montos extremos** (`total_amount > $1000`): posibles errores de facturación o registros de prueba del sistema.
- **Duración cero** (`dropoff == pickup`): cancelaciones inmediatas o fallos en el registro del tiempo de llegada.
- **Velocidad imposible** (> 150 mph): combinación de distancias largas y duraciones muy cortas, físicamente imposibles en NYC.

En los análisis anteriores estos registros fueron filtrados mediante condiciones de calidad (distancia > 0, montos > 0, duración entre 0 y 120 min), minimizando su efecto sobre los promedios y tendencias calculados.